In [1]:
"""
This code loads earnings announcement data and historical stock price data.

It defines the observation length, future prediction horizon, return thresholds,
column positions, and desired price channels such as open, high, low, close,
adjusted close, and volume. The earnings and price dates are converted to
datetime format, and the earnings rows are shuffled for processing.

The build_multivariate_dataset_using_adjclose function creates samples around
each earnings event. For each valid ticker and earnings date, it determines the
effective trading date based on whether the earnings release happened before or
after the market opened. It then extracts:
- an observed price window before the earnings event,
- a future price window after the earnings event,
- metadata containing the ticker and event date.

The function also resolves price column names flexibly, handles missing or
invalid data, skips samples without enough historical or future coverage, and
uses tqdm to show progress while processing earnings dates.

After all valid samples are collected, the code calculates the net future return
using the adjusted close or close price. Each sample is assigned a three-class
label:
- 1 for an upward move above the positive threshold,
- -1 for a downward move below the negative threshold,
- 0 for a relatively flat move between the thresholds.

Finally, the code returns the observed windows, future sequences, labels,
metadata, and resolved channel order. It then prints the total number of
samples, class counts, array shapes, and the final channel ordering.
"""
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
from sklearn.linear_model import RidgeClassifierCV
from tqdm import tqdm
from sklearn.preprocessing import label_binarize
from aeon.classification.convolution_based import MultiRocketHydraClassifier
# https://www.kaggle.com/datasets/tsaustin/us-historical-stock-prices-with-earnings-data
FILE_EARNINGS = '/Users/user/Downloads/earnings_latest.csv'
FILE_PRICES   = '/Users/user/Downloads/stock_prices_latest.csv'
OBS_LEN       = 40
FUTURE_LEN    = 5
UP_THRESH     = 0.01
DOWN_THRESH   = -0.01
PRICE_TICKER_COL = 0
PRICE_DATE_COL   = 1
EARN_TICKER_COL = 0
EARN_DATE_COL   = 1
EARN_TIME_COL   = 5  
DESIRED_CHANNELS = ("open", "high", "low", "close", "close_adjusted", "volume")
df = pd.read_csv(FILE_EARNINGS)
prices = pd.read_csv(FILE_PRICES)
df.iloc[:, EARN_DATE_COL] = pd.to_datetime(df.iloc[:, EARN_DATE_COL])
prices.iloc[:, PRICE_DATE_COL] = pd.to_datetime(prices.iloc[:, PRICE_DATE_COL])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
def build_multivariate_dataset_using_adjclose(
    df,
    prices,
    OBS_LEN,
    FUTURE_LEN,
    UP_THRESH,
    DOWN_THRESH,
    PRICE_TICKER_COL,
    PRICE_DATE_COL,
    EARN_TICKER_COL,
    EARN_DATE_COL,
    EARN_TIME_COL,
    desired_channels=DESIRED_CHANNELS,
    skip_on_nan=True,
):
    def _norm(s):
        return str(s).lower().replace(" ", "").replace("_", "").replace("-", "")
    def _match_col(cols, target):
        t = _norm(target)
        for c in cols:
            if _norm(c) == t:
                return c
        for c in cols:
            k = _norm(c)
            if t in k or k in t:
                return c
        return None
    price_cols = list(prices.columns)
    ticker_col_name = prices.columns[PRICE_TICKER_COL]
    date_col_name = prices.columns[PRICE_DATE_COL]
    resolved_channels = []
    for ch in desired_channels:
        found = _match_col(price_cols, ch)
        if found is None:
            raise RuntimeError(f"Could not find expected price column matching '{ch}' in: {price_cols}")
        resolved_channels.append(found)
    close_pref_candidates = ["close_adjusted", "adjclose", "adj_close", "adjustedclose", "close"]
    close_match = None
    for cand in close_pref_candidates:
        c = _match_col(price_cols, cand)
        if c is not None and c in resolved_channels:
            close_match = c
            break
    if close_match is None:
        raise RuntimeError("Could not locate an adjusted/close column among resolved channels for label calculation.")
    ticker_prices_dict = {ticker: group.reset_index(drop=True) for ticker, group in prices.groupby(prices.columns[PRICE_TICKER_COL])}
    observed_windows = []
    future_windows = []
    meta_rows = []
    with tqdm(total=len(df), desc="Processing earnings dates") as pbar:
        for _, row in df.iterrows():
            ticker = row.iloc[EARN_TICKER_COL]
            earnings_date = row.iloc[EARN_DATE_COL]
            release_time = row.iloc[EARN_TIME_COL]
            if pd.isna(release_time) or release_time not in ("pre", "post"):
                pbar.update(1); continue
            t0_calendar = earnings_date if release_time == "pre" else (earnings_date + pd.Timedelta(days=1))
            tp = ticker_prices_dict.get(ticker)
            if tp is None or tp.empty:
                pbar.update(1); continue
            try:
                tdates = tp[date_col_name].values.astype("datetime64[ns]")
            except Exception:
                tp = tp.copy()
                tp[date_col_name] = pd.to_datetime(tp[date_col_name])
                tdates = tp[date_col_name].values.astype("datetime64[ns]")
            idxs = np.where(tdates >= np.datetime64(t0_calendar))[0]
            if idxs.size == 0:
                pbar.update(1); continue
            t0_idx = int(idxs[0])
            if t0_idx - OBS_LEN < 0 or t0_idx + FUTURE_LEN >= len(tp):
                pbar.update(1); continue
            try:
                feat_df = tp.loc[:, resolved_channels].astype(float).reset_index(drop=True)
            except Exception:
                pbar.update(1); continue
            feats = feat_df.values  
            observed = feats[t0_idx - OBS_LEN : t0_idx]                       
            future_seq = feats[t0_idx : t0_idx + FUTURE_LEN + 1]              

            if observed.shape[0] != OBS_LEN or future_seq.shape[0] != (FUTURE_LEN + 1):
                pbar.update(1); continue
            if skip_on_nan and (np.isnan(observed).any() or np.isnan(future_seq).any()):
                pbar.update(1); continue
            observed_windows.append(observed)
            future_windows.append(future_seq)
            meta_rows.append((ticker, pd.Timestamp(tdates[t0_idx]).normalize()))
            pbar.update(1)
    if len(observed_windows) == 0:
        raise RuntimeError("No valid (observed, future) pairs were found. Check column indices and data coverage.")
    X_observed = np.array(observed_windows)            
    future_windows = np.array(future_windows)          
    meta_rows = np.array(meta_rows, dtype=object)
    close_idx = resolved_channels.index(close_match)
    t0_close = future_windows[:, 0, close_idx]
    tF_close = future_windows[:, FUTURE_LEN, close_idx]
    with np.errstate(divide='ignore', invalid='ignore'):
        net_ret = np.where(t0_close != 0, (tF_close - t0_close) / t0_close, 0.0)
    labels = np.zeros_like(net_ret, dtype=int)
    labels[net_ret >= UP_THRESH] = 1
    labels[net_ret <= DOWN_THRESH] = -1
    Y_future_seq = future_windows[:, 1:, :] 
    return X_observed, Y_future_seq, labels, meta_rows, resolved_channels
X_observed, Y_future_seq, labels, meta_rows, resolved_channels = build_multivariate_dataset_using_adjclose(
    df=df,
    prices=prices,
    OBS_LEN=OBS_LEN,
    FUTURE_LEN=FUTURE_LEN,
    UP_THRESH=UP_THRESH,
    DOWN_THRESH=DOWN_THRESH,
    PRICE_TICKER_COL=PRICE_TICKER_COL,
    PRICE_DATE_COL=PRICE_DATE_COL,
    EARN_TICKER_COL=EARN_TICKER_COL,
    EARN_DATE_COL=EARN_DATE_COL,
    EARN_TIME_COL=EARN_TIME_COL,
    desired_channels=DESIRED_CHANNELS,
    skip_on_nan=True,
)
n = labels.shape[0]
print(f"Total samples: {n}")
print(f"Class counts ->  -1 (down): {int((labels == -1).sum())}   0 (flat): {int((labels == 0).sum())}   1 (up): {int((labels == 1).sum())}")
print("X_observed shape:", X_observed.shape)      
print("Y_future_seq shape:", Y_future_seq.shape)  
print("labels shape:", labels.shape)
print("channel order (resolved):", resolved_channels)

/opt/anaconda3/lib/python3.12/site-packages/aeon/base/__init__.py:24: FutureWarning: The aeon package will soon be releasing v1.0.0 with the removal of legacy modules and interfaces such as BaseTransformer and BaseForecaster. This will contain breaking changes. See aeon-toolkit.org for more information. Set aeon.AEON_DEPRECATION_WARNING or the AEON_DEPRECATION_WARNING environmental variable to 'False' to disable this warning.
  warnings.warn(
Processing earnings dates: 100%|███████| 168603/168603 [09:22<00:00, 299.93it/s]


Total samples: 30928
Class counts ->  -1 (down): 14484   0 (flat): 4320   1 (up): 12124
X_observed shape: (30928, 40, 6)
Y_future_seq shape: (30928, 5, 6)
labels shape: (30928,)
channel order (resolved): ['open', 'high', 'low', 'close', 'close_adjusted', 'volume']


In [2]:
"""
This code chronologically splits the prepared stock-event time-series dataset
into training and testing sets.

It first tries to extract the earnings event dates from meta_rows and uses them
to sort all samples in time order. If the dates cannot be parsed, it falls back
to the existing sample order. The observed windows, future sequences, and labels
are then reordered consistently using this ordering.

The dataset is split into training and testing portions using TEST_SIZE, with
the earliest samples used for training and the latest samples used for testing.
The labels are shifted by +1 so that the original classes:
- -1 for downward movement,
- 0 for flat movement,
- 1 for upward movement,

become class indices 0, 1, and 2, which are more suitable for standard
multi-class classification models.

The code then prints the shapes of the training and testing arrays to verify
the split. It also selects the best available PyTorch compute device,
prioritising Apple Silicon MPS, then CUDA, and finally CPU.

Finally, the to_int helper function ensures that labels are represented as
integer class labels. If the labels are one-hot encoded, it converts them using
argmax; otherwise, it casts the existing labels directly to integers.
"""
TEST_SIZE = 0.2
try:
    t0_dates = pd.to_datetime([d for _, d in meta_rows])
    order = np.argsort(t0_dates.values)
except Exception:
    order = np.arange(labels.shape[0])
X_obs_all  = X_observed[order]    
X_fore_all = Y_future_seq[order]  
y_all      = labels[order]        
N = y_all.shape[0]
split_idx = int(np.floor(N * (1 - TEST_SIZE)))
X_observed_train = X_obs_all[:split_idx]
X_forecast_train  = X_fore_all[:split_idx]
y_train       = y_all[:split_idx] +1 
X_observed_test  = X_obs_all[split_idx:]
X_forecast_test  = X_fore_all[split_idx:]
y_test       = y_all[split_idx:] +1
print("Train shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes: ", X_observed_test.shape,  X_forecast_test.shape,  y_test.shape)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")
def to_int(y):
    y = np.asarray(y)
    return np.argmax(y, axis=1) if (y.ndim == 2 and y.shape[1] > 1) else y.astype(int)  

Train shapes: (24742, 40, 6) (24742, 5, 6) (24742,)
Test shapes:  (6186, 40, 6) (6186, 5, 6) (6186,)
Using device: mps


In [4]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        n_classes = proba.shape[1]
        classes = np.arange(n_classes)
        auroc_w = roc_auc_score(y_true,proba,multi_class="ovr",average="weighted")
        y_true_bin = label_binarize(y_true, classes=classes)
        auprc_w = average_precision_score(y_true_bin,proba,average="weighted")
        print(f"Weighted AUROC (seed {seed}): {auroc_w:.4f}")
        print(f"Weighted AUPRC (seed {seed}): {auprc_w:.4f}")
        results.append({"seed": seed,"auroc_weighted": auroc_w,"auprc_weighted": auprc_w,})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc_weighted']:.4f}  "
        f"AUPRC={r['auprc_weighted']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 36.00it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.4327    0.9133    0.5872      2677
           1     0.0816    0.0045    0.0085       890
           2     0.4600    0.0855    0.1442      2619

    accuracy                         0.4321      6186
   macro avg     0.3248    0.3345    0.2467      6186
weighted avg     0.3938    0.4321    0.3164      6186

Weighted AUROC (seed 0): 0.4961
Weighted AUPRC (seed 0): 0.3874

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 35.03it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.4324    0.8965    0.5834      2677
           1     0.0625    0.0034    0.0064       890
           2     0.4378    0.0981    0.1603      2619

    accuracy                         0.4300      6186
   macro avg     0.3109    0.3327    0.2500      6186
weighted avg     0.3815    0.4300    0.3213      6186

Weighted AUROC (seed 1): 0.5021
Weighted AUPRC (seed 1): 0.3925

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.4345    0.9212    0.5904      2677
           1     0.1356    0.0090    0.0169       890
           2     0.4678    0.0806    0.1375      2619

    accuracy                         0.4340      6186
   macro avg     0.3460    0.3369    0.2483      6186
weighted avg     0.4056    0.4340    0.3161      6186

Weighted AUROC (seed 2): 0.4977
Weighted AUPRC (seed 2): 0.3896

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 35.89it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.4372    0.8920    0.5868      2677
           1     0.1458    0.0079    0.0149       890
           2     0.4896    0.1264    0.2009      2619

    accuracy                         0.4407      6186
   macro avg     0.3576    0.3421    0.2675      6186
weighted avg     0.4175    0.4407    0.3411      6186

Weighted AUROC (seed 3): 0.5067
Weighted AUPRC (seed 3): 0.3966

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 35.60it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.4347    0.9227    0.5910      2677
           1     0.1250    0.0045    0.0087       890
           2     0.4555    0.0821    0.1391      2619

    accuracy                         0.4347      6186
   macro avg     0.3384    0.3364    0.2463      6186
weighted avg     0.3990    0.4347    0.3159      6186

Weighted AUROC (seed 4): 0.4967
Weighted AUPRC (seed 4): 0.3856

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.4961  AUPRC=0.3874
Seed 1: AUROC=0.5021  AUPRC=0.3925
Seed 2: AUROC=0.4977  AUPRC=0.3896
Seed 3: AUROC=0.5067  AUPRC=0.3966
Seed 4: AUROC=0.4967  AUPRC=0.3856


In [5]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        n_classes = proba.shape[1]
        classes = np.arange(n_classes)
        auroc_w = roc_auc_score(y_true,proba, multi_class="ovr",average="weighted")
        y_true_bin = label_binarize(y_true, classes=classes)  
        auprc_w = average_precision_score(y_true_bin,proba,average="weighted")
        print(f"Weighted AUROC (seed {seed}): {auroc_w:.4f}")
        print(f"Weighted AUPRC (seed {seed}): {auprc_w:.4f}")
        results.append({"seed": seed,"auroc_weighted": auroc_w,"auprc_weighted": auprc_w})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
    print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc_weighted']:.4f}  "
        f"AUPRC={r['auprc_weighted']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 49/49 [00:02<00:00, 23.96it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 35.64it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.4349    0.9208    0.5908      2677
           1     0.1739    0.0090    0.0171       890
           2     0.4534    0.0817    0.1385      2619

    accuracy                         0.4344      6186
   macro avg     0.3541    0.3372    0.2488      6186
weighted avg     0.4052    0.4344    0.3167      6186

Weighted AUROC (seed 0): 0.5018
Weighted AUPRC (seed 0): 0.3911

################################################################################

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.4335    0.9331    0.5919      2677
           1     0.0952    0.0045    0.0086       890
           2     0.4593    0.0668    0.1167      2619

    accuracy                         0.4328      6186
   macro avg     0.3293    0.3348    0.2391      6186
weighted avg     0.3957    0.4328    0.3068      6186

Weighted AUROC (seed 3): 0.4990
Weighted AUPRC (seed 3): 0.3895

################################################################################

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 49/49 [00:02<00:00, 23.43it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 49/49 [00:01<00:00, 35.47it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.4347    0.9227    0.5910      2677
           1     0.1053    0.0090    0.0166       890
           2     0.4720    0.0771    0.1326      2619

    accuracy                         0.4332      6186
   macro avg     0.3373    0.3363    0.2467      6186
weighted avg     0.4031    0.4332    0.3143      6186

Weighted AUROC (seed 4): 0.4985
Weighted AUPRC (seed 4): 0.3908

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.5018  AUPRC=0.3911
Seed 1: AUROC=0.4986  AUPRC=0.3915
Seed 2: AUROC=0.5031  AUPRC=0.3928
Seed 3: AUROC=0.4990  AUPRC=0.3895
Seed 4: AUROC=0.4985  AUPRC=0.3908


In [6]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        n_classes = mean_proba.shape[1]
        classes = np.arange(n_classes)
        auroc_weighted = roc_auc_score(y_true,mean_proba,multi_class="ovr",average="weighted")
        y_true_bin = label_binarize(y_true, classes=classes)  
        auprc_weighted = average_precision_score(y_true_bin,mean_proba,average="weighted")
        print(f"Weighted AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc_weighted:.4f}")
        print(f"Weighted AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc_weighted:.4f}")
        results.append({
            "seed": seed,
            "auroc_weighted": auroc_weighted,
            "auprc_weighted": auprc_weighted,
        })
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc_weighted']:.4f}  "
        f"AUPRC={r['auprc_weighted']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 49/49 [00:11<00:00,  4.18it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.4328    0.9701    0.5986      2677
           1     0.0909    0.0011    0.0022       890
           2     0.4400    0.0294    0.0551      2619

    accuracy                         0.4324      6186
   macro avg     0.3212    0.3335    0.2186      6186
weighted avg     0.3867    0.4324    0.2827      6186

------------------------------------------------------------------------
Weighted AUROC (MC mean over 100 samples, seed 0) : 0.5083
Weighted AUPRC (MC mean over 100 samples, seed 0) : 0.3961

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 49/49 [00:11<00:00,  4.20it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.4327    0.9589    0.5963      2677
           1     0.2500    0.0011    0.0022       890
           2     0.4538    0.0431    0.0788      2619

    accuracy                         0.4334      6186
   macro avg     0.3788    0.3344    0.2258      6186
weighted avg     0.4153    0.4334    0.2917      6186

------------------------------------------------------------------------
Weighted AUROC (MC mean over 100 samples, seed 1) : 0.4990
Weighted AUPRC (MC mean over 100 samples, seed 1) : 0.3896

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.4340    0.9585    0.5975      2677
           1     0.0667    0.0011    0.0022       890
           2     0.4981    0.0493    0.0896      2619

    accuracy                         0.4358      6186
   macro avg     0.3329    0.3363    0.2298      6186
weighted avg     0.4083    0.4358    0.2968      6186

------------------------------------------------------------------------
Weighted AUROC (MC mean over 100 samples, seed 2) : 0.5092
Weighted AUPRC (MC mean over 100 samples, seed 2) : 0.3978

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 49/49 [00:11<00:00,  4.21it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.4333    0.9705    0.5991      2677
           1     0.2222    0.0022    0.0044       890
           2     0.4530    0.0313    0.0586      2619

    accuracy                         0.4336      6186
   macro avg     0.3695    0.3347    0.2207      6186
weighted avg     0.4113    0.4336    0.2847      6186

------------------------------------------------------------------------
Weighted AUROC (MC mean over 100 samples, seed 3) : 0.5060
Weighted AUPRC (MC mean over 100 samples, seed 3) : 0.3935

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 49/49 [00:11<00:00,  4.20it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.4325    0.9653    0.5974      2677
           1     0.1250    0.0022    0.0044       890
           2     0.4439    0.0332    0.0618      2619

    accuracy                         0.4321      6186
   macro avg     0.3338    0.3336    0.2212      6186
weighted avg     0.3931    0.4321    0.2853      6186

------------------------------------------------------------------------
Weighted AUROC (MC mean over 100 samples, seed 4) : 0.5064
Weighted AUPRC (MC mean over 100 samples, seed 4) : 0.3950

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.5083  AUPRC=0.3961
Seed 1: AUROC=0.4990  AUPRC=0.3896
Seed 2: AUROC=0.5092  AUPRC=0.3978
Seed 3: AUROC=0.5060  AUPRC=0.

In [4]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        n_classes = proba.shape[1]
        classes = np.arange(n_classes)
        auroc_weighted = roc_auc_score(y_true,proba, multi_class="ovr",average="weighted")
        y_true_bin = label_binarize(y_true, classes=classes)  
        auprc_weighted = average_precision_score(y_true_bin,proba,average="weighted")
        print(f"Weighted AUROC (seed {seed}): {auroc_weighted:.4f}")
        print(f"Weighted AUPRC (seed {seed}): {auprc_weighted:.4f}")
        results.append({"seed": seed,"auroc_weighted": auroc_weighted,"auprc_weighted": auprc_weighted,})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )


Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.4338    0.4285    0.4311      2677
           1     0.1729    0.1348    0.1515       890
           2     0.4249    0.4620    0.4427      2619

    accuracy                         0.4004      6186
   macro avg     0.3439    0.3418    0.3418      6186
weighted avg     0.3925    0.4004    0.3958      6186

Weighted AUROC (seed 0): 0.5029
Weighted AUPRC (seed 0): 0.3883

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.4284    0.4270    0.4277      2677
           1     0.1725    0.1337    0.1506       890
           2     0.4240    0.4578    0.4402      2619

    accuracy                         0.3978      6186
   macro avg     0.3416    0.3395    0.3395      6186
weighted avg     0.3897    0.3978    0.3931      6186

Weighted AUROC (

NameError: name 'MC_SAMPLES' is not defined